# Storage Accuracy Audit

This notebook checks whether extracted paper data is being stored and retrievable from PostgreSQL and Qdrant. It compares live counts, then audits one stored document by reading the persisted rows and vector payloads back.

In [1]:
from pathlib import Path
import json
import os
import sys
from collections import Counter

from dotenv import load_dotenv
from sqlalchemy import text
from qdrant_client.models import FieldCondition, Filter, MatchValue

def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / '.env').exists():
            return candidate
    return start

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / '.env')

from backend.database.connection import DatabaseConnection
from backend.extraction.persistence.postgres_store import (
    ImageRecord,
    PaperRecord,
    SectionRecord,
    TableDataRecord,
    TextBlockRecord,
)
from rag.retrieval.pipeline import RetrievalPipeline

print(f'Project root: {ROOT}')
print(f'DATABASE_URL set: {bool(os.getenv("DATABASE_URL"))}')
print(f'QDRANT_URL set: {bool(os.getenv("QDRANT_URL"))}')

/home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration loaded:
  - API Host: 0.0.0.0:8000
  - Upload Directory: uploads
  - Max File Size: 50 MB
  - Groq API Configured: True
  - Qdrant Configured: True
  - Qdrant Collection: research_papers
  - LangSmith Tracing: Enabled
Project root: /home/aman/storage/Python/Projects/Research Paper Assistant
DATABASE_URL set: True
QDRANT_URL set: True


In [2]:
db = DatabaseConnection()
db.create_tables()
pipeline = RetrievalPipeline(enable_reranking=False)
store_manager = pipeline._get_store_manager()
client = store_manager.client

def pg_counts() -> dict[str, int]:
    tables = ['papers', 'sections', 'text_blocks', 'tables_data', 'images', 'references_data']
    with db.session() as sess:
        return {table: sess.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar_one() for table in tables}

def qdrant_count() -> int:
    return client.count(collection_name=store_manager.collection_name, exact=True).count

print('PostgreSQL counts:')
for table, count in pg_counts().items():
    print(f'  {table}: {count}')

print('Qdrant counts:')
print(f'  collection: {store_manager.collection_name}')
print(f'  points: {qdrant_count()}')
print(f'  collection_info: {store_manager.get_collection_info()}')

INFO:backend.database.connection:DatabaseConnection initialised (url masked for security)
INFO:backend.database.connection:Database tables created / verified.
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
INFO:rag.retrieval.indexing.qdrant_store:QdrantStoreManager: connected to cloud at https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333


PostgreSQL counts:
  papers: 12
  sections: 84
  text_blocks: 1713
  tables_data: 45
  images: 147
  references_data: 560
Qdrant counts:
  collection: research_papers


INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/count "HTTP/1.1 200 OK"


  points: 1941


INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"


  collection_info: {'name': 'research_papers', 'status': 'green', 'vectors_count': None, 'points_count': 1941}


In [6]:
def normalize(text_value: str | None) -> str:
    return ' '.join(str(text_value or '').strip().lower().split())


def scroll_all_points(document_id: str, page_size: int = 100):
    points = []
    offset = None
    scroll_filter = Filter(
        must=[FieldCondition(key='document_id', match=MatchValue(value=document_id))]
    )
    while True:
        batch, offset = client.scroll(
            collection_name=store_manager.collection_name,
            scroll_filter=scroll_filter,
            limit=page_size,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )
        points.extend(batch)
        if not batch or offset is None:
            break
    return points


def audit_latest_paper(sample_limit: int = 10) -> dict:
    with db.session() as sess:
        paper = sess.query(PaperRecord).order_by(PaperRecord.created_at.desc()).first()
        if paper is None:
            return {'passed': False, 'mismatches': ['no paper rows found']}

        sections = sess.query(SectionRecord).filter(SectionRecord.paper_id == paper.id).all()
        text_blocks = sess.query(TextBlockRecord).filter(TextBlockRecord.paper_id == paper.id).all()
        tables = sess.query(TableDataRecord).filter(TableDataRecord.paper_id == paper.id).all()
        images = sess.query(ImageRecord).filter(ImageRecord.paper_id == paper.id).all()

    document_id = paper.document_uuid or str(paper.id)
    qdrant_points = scroll_all_points(document_id)

    mismatches = []
    title_observations = []
    if not qdrant_points:
        mismatches.append(f'no Qdrant points found for document_id={document_id}')

    expected_section_titles = {normalize(section.original_name) for section in sections}
    expected_section_titles.update({'reference', 'references'})
    expected_section_prefix = f'{document_id}_section_'

    for point in qdrant_points[:sample_limit]:
        payload = point.payload or {}
        payload_document_id = payload.get('document_id')
        payload_chunk_id = payload.get('chunk_id')
        payload_content = payload.get('content')
        payload_section_id = payload.get('section_id')
        payload_section_title = payload.get('section_title')
        normalized_title = normalize(payload_section_title)

        if payload_document_id != document_id:
            mismatches.append(f'point {point.id} has document_id={payload_document_id} expected={document_id}')
        if not payload_chunk_id:
            mismatches.append(f'point {point.id} is missing chunk_id')
        if not payload_content:
            mismatches.append(f'point {point.id} is missing content')
        if not str(payload_section_id or '').startswith(expected_section_prefix):
            mismatches.append(f'point {point.id} has malformed section_id={payload_section_id}')
        if normalized_title and normalized_title not in expected_section_titles:
            title_observations.append(
                f'point {point.id} has section_title={payload_section_title} not found in Postgres sections'
            )

    return {
        'passed': not mismatches,
        'paper_id': paper.id,
        'paper_name': paper.paper_name,
        'document_id': document_id,
        'postgres': {
            'sections': len(sections),
            'text_blocks': len(text_blocks),
            'tables': len(tables),
            'images': len(images),
        },
        'qdrant_points': len(qdrant_points),
        'sample_payload_keys': sorted(list((qdrant_points[0].payload or {}).keys())) if qdrant_points else [],
        'mismatches': mismatches,
        'title_observations': title_observations,
    }


audit = audit_latest_paper()
print(json.dumps(audit, indent=2, default=str))

INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/scroll "HTTP/1.1 200 OK"


{
  "passed": true,
  "paper_id": 74,
  "paper_name": "Explaining Context Length Scaling and Bounds for Language Models",
  "document_id": "646ce3f5-927d-5538-a4dd-2c9a68680990",
  "postgres": {
    "sections": 7,
    "text_blocks": 181,
    "tables": 1,
    "images": 14
  },
  "qdrant_points": 254,
  "sample_payload_keys": [
    "chunk_id",
    "chunk_index",
    "chunk_level",
    "content",
    "content_type",
    "document_id",
    "element_ids",
    "image_path",
    "original_content",
    "page_end",
    "page_start",
    "parent_section_id",
    "section_id",
    "section_level",
    "section_numbering",
    "section_path",
    "section_path_ids",
    "section_title",
    "source_file",
    "token_count"
  ],
  "mismatches": [],
  "title_observations": [
    "point 03992c1b-5bcc-5928-86fe-b460246df8b5 has section_title=Conclusion and Discussions not found in Postgres sections",
    "point 03ee5dd0-b341-54c7-a506-8ab9de880caf has section_title=Proof of Concept with Synthetic Dat

In [8]:
summary = 'PASS' if audit['passed'] else 'FAIL'
print(f'Storage audit result: {summary}')
if audit['mismatches']:
    print('Mismatches:')
    for item in audit['mismatches']:
        print(f'  - {item}')
else:
    print('No mismatches found in the sampled document and payloads.')

Storage audit result: PASS
No mismatches found in the sampled document and payloads.
